# 02. Chat Model 호출

LangChain에서 OpenAI ChatModel을 불러오고 기본 호출 방법을 실습합니다.

## 1. 환경 설정 
## 2. ChatOpenAI 초기화

- `model`: 사용할 모델 이름
- `temperature`: 0에 가까울수록 일관된 답변, 1에 가까울수록 창의적인 답변

In [1]:
# 상위 폴더 path 설정
import sys
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_openai import ChatOpenAI
from common_config import llm_connect

temperature = 0
max_tokens = 2086

llm = llm_connect()

In [2]:
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, profile={'name': 'GPT-5.4 mini', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']}, client=<openai.resources.chat.completions.completions.Completions object at 0x761783bdf860>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7617839

## 3. 기본 호출 - invoke()

문자열을 직접 전달하면 HumanMessage로 자동 변환됩니다.

In [3]:
response = llm.invoke("LangChain이 무엇인지 한 문장으로 설명해줘")
print(response)

content='LangChain은 **대규모 언어 모델(LLM)을 활용한 앱을 쉽게 만들 수 있도록 도와주는 프레임워크**입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 18, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini', 'system_fingerprint': None, 'id': 'chatcmpl-EU3GzmNNQMAHIh5oZYjNIC1CEkecT', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0f5da-7771-74f2-bd11-cc85bbac1640-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 35, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [4]:
# content만 출력
print(response.content)

LangChain은 **대규모 언어 모델(LLM)을 활용한 앱을 쉽게 만들 수 있도록 도와주는 프레임워크**입니다.


## 4. 메시지 객체로 호출

역할(role)을 구분하여 메시지를 전달할 수 있습니다.
- `SystemMessage`: AI의 역할/페르소나 설정
- `HumanMessage`: 사용자 입력

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 친절한 파이썬 튜터입니다."),
    HumanMessage(content="리스트 컴프리헨션을 초보자에게 설명해줘")
]

response = llm.invoke(messages)
print(response.content)

물론이죠!  
**리스트 컴프리헨션(list comprehension)**은 파이썬에서 **리스트를 짧고 읽기 쉽게 만드는 문법**입니다.

---

## 1) 기본 아이디어

원래는 이런 식으로 리스트를 만들 수 있어요:

```python
numbers = [1, 2, 3, 4, 5]
squares = []

for n in numbers:
    squares.append(n * n)

print(squares)
# [1, 4, 9, 16, 25]
```

이걸 리스트 컴프리헨션으로 쓰면:

```python
numbers = [1, 2, 3, 4, 5]
squares = [n * n for n in numbers]

print(squares)
# [1, 4, 9, 16, 25]
```

더 짧고 간단하죠.

---

## 2) 기본 구조

리스트 컴프리헨션의 형태는 보통 이렇게 생겼어요:

```python
[표현식 for 변수 in 반복가능한객체]
```

예:

```python
[n * 2 for n in [1, 2, 3]]
```

뜻:
- 리스트 `[1, 2, 3]`의 각 값 `n`에 대해
- `n * 2`를 계산해서
- 새로운 리스트를 만든다

결과:

```python
[2, 4, 6]
```

---

## 3) 조건도 넣을 수 있어요

짝수만 골라서 제곱하고 싶다면:

```python
numbers = [1, 2, 3, 4, 5, 6]
even_squares = [n * n for n in numbers if n % 2 == 0]

print(even_squares)
# [4, 16, 36]
```

구조는:

```python
[표현식 for 변수 in 반복가능한객체 if 조건]
```

뜻:
- 조건을 만족하는 값만 골라서
- 그 값으로 표현식을 실행

---

## 4) if와 for의 순서가 중요해요

이 코드는:

```python
[n * n for n in numbers if n % 2 == 0]
```

다음 의미예요:


- 마크다운으로 출력해서 보기

In [6]:
from IPython.display import Markdown, display
display(Markdown(response.content))

물론이죠!  
**리스트 컴프리헨션(list comprehension)**은 파이썬에서 **리스트를 짧고 읽기 쉽게 만드는 문법**입니다.

---

## 1) 기본 아이디어

원래는 이런 식으로 리스트를 만들 수 있어요:

```python
numbers = [1, 2, 3, 4, 5]
squares = []

for n in numbers:
    squares.append(n * n)

print(squares)
# [1, 4, 9, 16, 25]
```

이걸 리스트 컴프리헨션으로 쓰면:

```python
numbers = [1, 2, 3, 4, 5]
squares = [n * n for n in numbers]

print(squares)
# [1, 4, 9, 16, 25]
```

더 짧고 간단하죠.

---

## 2) 기본 구조

리스트 컴프리헨션의 형태는 보통 이렇게 생겼어요:

```python
[표현식 for 변수 in 반복가능한객체]
```

예:

```python
[n * 2 for n in [1, 2, 3]]
```

뜻:
- 리스트 `[1, 2, 3]`의 각 값 `n`에 대해
- `n * 2`를 계산해서
- 새로운 리스트를 만든다

결과:

```python
[2, 4, 6]
```

---

## 3) 조건도 넣을 수 있어요

짝수만 골라서 제곱하고 싶다면:

```python
numbers = [1, 2, 3, 4, 5, 6]
even_squares = [n * n for n in numbers if n % 2 == 0]

print(even_squares)
# [4, 16, 36]
```

구조는:

```python
[표현식 for 변수 in 반복가능한객체 if 조건]
```

뜻:
- 조건을 만족하는 값만 골라서
- 그 값으로 표현식을 실행

---

## 4) if와 for의 순서가 중요해요

이 코드는:

```python
[n * n for n in numbers if n % 2 == 0]
```

다음 의미예요:
1. `numbers`를 하나씩 꺼내고
2. `n % 2 == 0`인지 확인한 뒤
3. 맞으면 `n * n`을 리스트에 넣기

---

## 5) 예제 몇 가지

### 예제 1: 문자열 길이 구하기

```python
words = ["cat", "python", "hi"]
lengths = [len(word) for word in words]

print(lengths)
# [3, 6, 2]
```

### 예제 2: 대문자로 바꾸기

```python
words = ["apple", "banana", "grape"]
upper_words = [word.upper() for word in words]

print(upper_words)
# ['APPLE', 'BANANA', 'GRAPE']
```

### 예제 3: 3의 배수만 선택하기

```python
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9]
multiples_of_3 = [n for n in nums if n % 3 == 0]

print(multiples_of_3)
# [3, 6, 9]
```

---

## 6) 언제 쓰면 좋을까?

리스트 컴프리헨션은 다음 경우에 좋습니다:

- 간단한 반복으로 새 리스트를 만들 때
- 코드가 짧고 명확해질 때

하지만 너무 복잡하면 오히려 읽기 어려워질 수 있어요.  
그럴 땐 `for`문을 쓰는 게 더 좋습니다.

---

## 7) 초보자가 기억할 한 줄 요약

**“리스트의 각 요소를 바꿔서 새 리스트를 만들거나, 조건에 맞는 것만 골라낼 때 쓰는 문법”**

---

원하시면 다음으로  
**“for문과 리스트 컴프리헨션을 비교해서 쉽게 이해하기”** 또는  
**“중첩 리스트 컴프리헨션”**도 설명해드릴게요.

## 5. 스트리밍 호출 - stream()

- 토큰이 생성되는 즉시 출력함.
- llm 객체에 질문을 보내고, 응답을 한 번에 받지 않고 조각(chunk) 단위로 순차적으로 받음.
- end=""를 쓰면 줄바꿈 없이 이어서 출력함.
- flush=True는 출력 버퍼를 즉시 비우라는 의미임.

In [7]:
for chunk in llm.stream("최신 langchain 버전을 알여주고, 특장점을 알여줘 "):
    print(chunk.content, end="", flush=True)

제가 가진 정보 기준으로는 **LangChain의 최신 버전을 실시간으로 조회할 수는 없어서** 정확한 “현재 최신 버전 번호”를 바로 단정하긴 어렵습니다.  
다만 **최신 버전은 PyPI 기준으로 확인**하는 게 가장 정확합니다.

### 최신 버전 확인 방법
- **Python 패키지 설치 확인**
  ```bash
  pip index versions langchain
  ```
- 또는 PyPI에서 확인:
  - https://pypi.org/project/langchain/

---

## LangChain의 주요 특장점
LangChain은 **LLM(대규모 언어 모델) 기반 애플리케이션을 빠르게 만들 수 있게 해주는 프레임워크**입니다.

### 1) 체인(Chain) 구조로 복잡한 흐름 구성
여러 단계를 연결해서:
- 질문 분류
- 검색
- 요약
- 답변 생성  
같은 복합 작업을 쉽게 만들 수 있습니다.

### 2) 다양한 LLM/모델 연동
OpenAI, Anthropic, Google, Hugging Face 등 여러 모델과 연동하기 쉬워서  
**모델을 바꿔도 코드 변경이 적은 편**입니다.

### 3) RAG(검색 증강 생성) 구현이 편리
문서 검색 + LLM 응답 생성을 결합하는 **RAG 시스템**을 만들기 좋습니다.
- 벡터 DB 연동
- 문서 로딩
- 임베딩
- 검색
- 프롬프트 구성  
이런 구성요소를 잘 지원합니다.

### 4) 도구 사용(툴/에이전트) 지원
LLM이:
- API 호출
- 계산기 사용
- DB 조회
- 웹 검색  
같은 외부 도구를 사용하게 만드는 에이전트 구현이 가능합니다.

### 5) 오픈소스 생태계와 커뮤니티가 큼
예제, 문서, 통합 라이브러리가 많아서  
**프로토타입을 빠르게 만들고 학습하기 좋습니다.**

### 6) LangGraph 등과의 확장성
최근에는 복잡한 상태 기반 워크플로우를 위한 **LangGraph** 같은 도구와 함께 쓰는 경우가 많아,
**에이전트/멀티스텝 워크플로우**를 더

## 6. 응답 메타데이터 확인

토큰 사용량 등 메타정보를 확인할 수 있습니다.

In [8]:
response = llm.invoke("안녕")
print("content:", response.content)
print("usage:", response.usage_metadata)

content: 안녕하세요! 무엇을 도와드릴까요?
usage: {'input_tokens': 8, 'output_tokens': 14, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
